1. Kütüphane Yükleme ve Veri Seti Okuma:
Bu blokta, modelleme, veri ön işleme, boyut indirgeme (PCA) ve değerlendirme için gerekli tüm kütüphaneler yüklenir.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.decomposition import PCA

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier

from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [2]:
df = pd.read_csv("DataSet/googleplaystore.csv")
print(df.head())
print("Shape:", df.shape)

                                                 App        Category  Rating  \
0     Photo Editor & Candy Camera & Grid & ScrapBook  ART_AND_DESIGN     4.1   
1                                Coloring book moana  ART_AND_DESIGN     3.9   
2  U Launcher Lite – FREE Live Cool Themes, Hide ...  ART_AND_DESIGN     4.7   
3                              Sketch - Draw & Paint  ART_AND_DESIGN     4.5   
4              Pixel Draw - Number Art Coloring Book  ART_AND_DESIGN     4.3   

  Reviews  Size     Installs  Type Price Content Rating  \
0     159   19M      10,000+  Free     0       Everyone   
1     967   14M     500,000+  Free     0       Everyone   
2   87510  8.7M   5,000,000+  Free     0       Everyone   
3  215644   25M  50,000,000+  Free     0           Teen   
4     967  2.8M     100,000+  Free     0       Everyone   

                      Genres Last Updated         Current Ver   Android Ver  
0               Art & Design     7-Jan-18               1.0.0  4.0.3 and up  
1  Art &

2. Veri Temizleme Fonksiyonları ve Hedef Değişken Oluşturma:
Bu blokta, önceki çalışmalarda kullanılan aynı temizleme fonksiyonları tanımlanır, orijinal verilere uygulanır ve hedef değişken (High_Rating) oluşturulur.

In [3]:
def clean_size(s):
    if isinstance(s, str):
        s = s.strip()
        if s.endswith('M'):
            try:
                return float(s[:-1])
            except:
                return np.nan
        if s.endswith('k'):
            try:
                return float(s[:-1]) / 1024.0
            except:
                return np.nan
        if s == 'Varies with device':
            return np.nan
    return np.nan

def clean_installs(s):
    if isinstance(s, str):
        s = s.replace('+', '').replace(',', '')
        try:
            return int(s)
        except:
            return np.nan
    return np.nan

def clean_price(s):
    if isinstance(s, str):
        s = s.replace('$', '')
        try:
            return float(s)
        except:
            return np.nan
    return np.nan

df2 = df.copy()

df2['Size_Mb']        = df2['Size'].apply(clean_size)
df2['Installs_clean'] = df2['Installs'].apply(clean_installs)
df2['Price_clean']    = df2['Price'].apply(clean_price)


df2 = df2[~df2['Rating'].isna()].copy()


df2['High_Rating'] = (df2['Rating'] >= 4.0).astype(int)


df2['Reviews'] = pd.to_numeric(df2['Reviews'], errors='coerce')


3. Özellik ve Hedef Tanımlama:
Modelde kullanılacak son özellikler (X) belirlenir ve hedef değişkenin (y) dağılımı incelenir.

In [4]:
features = [
    'Category', 'Reviews', 'Size_Mb', 'Installs_clean',
    'Type', 'Price_clean', 'Content Rating', 'Genres'
]

X = df2[features].copy()
y = df2['High_Rating'].copy()

print("X shape:", X.shape)
print("Positive class ratio:", y.mean())

X shape: (9367, 8)
Positive class ratio: 0.7866979822782108


4. Veri Bölme:
Veri, eğitim ve test kümelerine ayrılır. Dengesiz veri setlerinde sınıf oranını korumak için stratify=y kullanılır.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (7493, 8) Test shape: (1874, 8)


5. Ön İşleme ve Özellik Azaltma Hazırlığı:
Bu blok, ön işleme adımlarını tanımlar ve boyut azaltma için PCA nesnesini hazırlar.

In [6]:
numeric_features = ['Reviews', 'Size_Mb', 'Installs_clean', 'Price_clean']
categorical_features = ['Category', 'Type', 'Content Rating', 'Genres']

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)


to_dense = FunctionTransformer(lambda x: x.toarray(), accept_sparse=True)


pca = PCA(n_components=30, random_state=42)

6. Model Pipeline'ları Oluşturma:
Pipeline'lar, ön işleme, seyrek-yoğun dönüşüm, PCA ile boyut azaltma ve son olarak sınıflandırıcı adımlarını içerir.

In [7]:
pipe_ahmed_nb = Pipeline(steps=[
    ("preprocess", preprocess),
    ("to_dense", to_dense),
    ("pca", pca),
    ("clf", GaussianNB())
])

pipe_ahmed_dt = Pipeline(steps=[
    ("preprocess", preprocess),
    ("to_dense", to_dense),
    ("pca", pca),
    ("clf", DecisionTreeClassifier(random_state=42))
])

pipe_ahmed_extra = Pipeline(steps=[
    ("preprocess", preprocess),
    ("to_dense", to_dense),
    ("pca", pca),
    ("clf", ExtraTreesClassifier(n_estimators=200, random_state=42))
])



7. Model Değerlendirmesi ve Sonuçlar:
run_model fonksiyonu, modelleri eğitir ve test seti üzerinde metrikleri (Classification Report ve ROC AUC) yazdırır.

In [8]:
def run_model(name, pipe):
    print("\n" + "="*90)
    print(f"START MODEL: {name}")
    print("="*90)

    print("Training model ...")
    pipe.fit(X_train, y_train)

    print("Predicting ...")
    y_pred = pipe.predict(X_test)

    print("Classification Report:")
    print(classification_report(y_test, y_pred))

    if hasattr(pipe, "predict_proba"):
        y_proba = pipe.predict_proba(X_test)[:, 1]
        print("ROC AUC:", roc_auc_score(y_test, y_proba))

    print("DONE:", name)
    print("="*90)


In [9]:
run_model("Ahmed - GaussianNB + PCA", pipe_ahmed_nb)
run_model("Ahmed - DecisionTree + PCA", pipe_ahmed_dt)
run_model("Ahmed - ExtraTrees + PCA", pipe_ahmed_extra)


START MODEL: Ahmed - GaussianNB + PCA
Training model ...
Predicting ...
Classification Report:
              precision    recall  f1-score   support

           0       0.29      0.60      0.39       400
           1       0.85      0.60      0.70      1474

    accuracy                           0.60      1874
   macro avg       0.57      0.60      0.55      1874
weighted avg       0.73      0.60      0.64      1874

ROC AUC: 0.6310142469470827
DONE: Ahmed - GaussianNB + PCA

START MODEL: Ahmed - DecisionTree + PCA
Training model ...
Predicting ...
Classification Report:
              precision    recall  f1-score   support

           0       0.37      0.39      0.38       400
           1       0.83      0.82      0.83      1474

    accuracy                           0.73      1874
   macro avg       0.60      0.61      0.60      1874
weighted avg       0.73      0.73      0.73      1874

ROC AUC: 0.6049940637720488
DONE: Ahmed - DecisionTree + PCA

START MODEL: Ahmed - ExtraTrees